# L01 · Optional assay pilot provenance and analysis

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Requires an analytical collaborator, actual laboratory data and separate authorisation. Nothing in this notebook specifies wet-lab procedures or invents concentrations.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Read actual assay results

In [ ]:
from oncoplate.governance import require_gate
from oncoplate.studies import assay_qc,assay_detected_sensitivity
require_gate(cfg,'lab_study')
lab_root=Path(cfg['root'])/'private/lab'
assays=read_table(lab_root/'assays.csv')
report=assay_qc(assays);write_json(p['reports']/'assay_pilot_qc.json',report);print(report)

## 2. Optional detected-only sensitivity comparison
Censored observations remain in the main data. This simple sensitivity analysis is not a replacement for a reviewed censored-outcome model.

In [ ]:
import numpy as np
with np.load(lab_root/'paired_features.npz',allow_pickle=False) as z:
    ids=z['specimen_ids'];image_features=z['image_features'];preparation_features=z['preparation_features']
assert ids.tolist()==assays.specimen_id.tolist(),'Specimen-feature alignment mismatch'
results=assay_detected_sensitivity(assays,image_features,preparation_features)
write_table(p['reports']/'assay_detected_only_sensitivity.csv',results);display(results)

## 3. Keep chemical validation separate from visual label agreement

In [ ]:
print('These outputs do not turn FoodNExTDB labels into chemical ground truth.')
print('Technical replicates and multiple photographs do not become independent biological specimens.')
print('Any primary chemical inference needs the collaborator-reviewed assay/censoring protocol and external validation.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
